# Klasifikasi LULC Jawa Timur: k-NN Scaled
**Joni Kumar Meghwar · NIM 240411100234**

Same sequence as the reference: pixel features → polygon means → 70/30 split → StandardScaler and k-NN → evaluation exports → satellite overlay. All results come from your own extraction. No results have been prefilled.

In [ ]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uts/code/pipeline.py").exists()), None)
if root is None: raise RuntimeError("Open this notebook inside the extracted project folder")
sys.path.insert(0, str(root / "uts/code"))
from pipeline import *

## 1. Business and data understanding
Goal: support preliminary land/coastal review by distinguishing water, buildings, agriculture, land forest and mangrove. The study uses digitized sample polygons, not every pixel in East Java.

In [ ]:
display(pd.read_csv(DEPLOY / "sample_summary.csv"))

## 2. Load real pixels and calculate spectral indices
Run crawling first. Indices are computed per valid pixel before aggregation. B8A is narrow NIR; the original B08 indices remain, and NDMI_8A is added.

In [ ]:
pixels = add_indices(pd.read_csv(UTS / "data/csv/hasil_s2_piksel/piksel_s2.csv"))
display(pixels[FEATURES].describe())
display(pixels.groupby("label_teks").size().rename("Pixels"))

## 3. EDA before modeling

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
means=pixels.groupby("poligon_id")[FEATURES].mean()
fig,ax=plt.subplots(figsize=(10,8))
sns.heatmap(means.corr(),cmap="RdBu_r",center=0,ax=ax)
plt.show()
fig,ax=plt.subplots(figsize=(10,5))
sns.boxplot(data=pixels,x="label_teks",y="NDMI_8A",ax=ax)
ax.tick_params(axis="x",rotation=20)
plt.show()

## 4. Train and evaluate
The pipeline splits at polygon level, fits the scaler only on training, uses k=5/distance/Euclidean and evaluates the held-out test set. Validation curves use training cross-validation. Test results are not used to select k. Invalid/missing polygons are reported, so final counts are calculated dynamically.

In [ ]:
train()

## 5. Evaluation and exported files

In [ ]:
for name in ["ringkasan_model_knn.csv","metrik_evaluasi_knn.csv","confusion_matrix.csv","kepentingan_fitur.csv","kurva_validasi_k.csv"]:
    print(name)
    display(pd.read_csv(DEPLOY / name))

In [ ]:
cm=pd.read_csv(DEPLOY / "confusion_matrix.csv",index_col=0)
fig,ax=plt.subplots(figsize=(8,6))
sns.heatmap(cm,annot=True,fmt="d",cmap="Oranges",ax=ax)
ax.set_xlabel("Predicted");ax.set_ylabel("Actual")
plt.show()
display(pd.read_csv(DEPLOY / "missing_satellite_polygons.csv"))

## 6. Satellite map and discussion
Predictions are shown only on held-out testing polygons. Training remains labeled as training. Discuss your actual confusion matrix, mangrove/forest errors, clouds and digitization quality. Random polygon splitting does not remove geographic autocorrelation; consider spatial blocks for stronger evidence.

In [ ]:
from IPython.display import IFrame
display(IFrame(str(DEPLOY / "map_lulc_predicted.html"), width="100%", height=650))

## 7. Deployment
Run `python -m streamlit run uts/deploy/app.py` from the root folder. Push generated CSVs, evaluation GeoJSON, polygon features and model files to your GitHub repository. Follow `uts/deployment-guide.md` and add your real Streamlit URL after deployment.